In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split

from src.utils import *
import src.prompt as prompt
from src.data_loader import load_spatial_data_onefile

from matplotlib import pyplot as plt
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['pdf.fonttype'] = 42

import openai
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
config = load_config("configs/config_finetunePro_copd.yaml")
config.data_name = "230267_Slide2"
config.replicate = "_test_2niche_50cells"
config.refresh_paths()


# data  

In [ ]:
data_path = str(dataset_dir("copd", "230267_Slide2"))
adata = load_spatial_data_onefile(f"{data_path}/{config.data_name}_zeroshot_gpt4o_refined_k20.csv", 
                                  config, 
                                  index_col=0)

# Prepare neighbor data using the new function
neighbor_normalized_df, neighbor_normalized_df_genes, adj_matrix = prepare_neighbor_data(
    adata, config
)

## extra: rename _i to _inflamed

In [ ]:
neighbor_normalized_df.columns = neighbor_normalized_df.columns.str.replace("_i", "_inflamed")

# labeled data for training  

In [ ]:
selected_cells = pd.read_csv(f"{data_path}/selected_cells/selected_cells_id.csv",index_col=0)
# add "Slide2_" to the index of selected_cells
selected_cells.index = "Slide2_" + selected_cells.index

# remove Unassigned in label column
selected_cells = selected_cells[selected_cells[config.name_truth] != "Unassigned"]



# # TODO: test with 50 cells in total to check whether only two niche are enough
# # random select 25 cells with niche value is Airway Inflamed 
# selected_cells_1 = selected_cells[selected_cells.niche == "Airway Inflamed"].sample(n=25, random_state=42)
# # random select 25 cells with niche value is Large Vessel Inflamed
# selected_cells_2 = selected_cells[selected_cells.niche == "Large Vessel Inflamed"].sample(n=25, random_state=42)
# selected_cells = pd.concat([selected_cells_1, selected_cells_2])
print(selected_cells.niche.value_counts())




adata.obs = adata.obs.join(selected_cells) 


In [ ]:
# seperate neighbor_normalized_df by selected_cells as train and test data
train_neighbor_normalized_df = neighbor_normalized_df[neighbor_normalized_df.index.isin(selected_cells.index)].copy()
test_neighbor_normalized_df = neighbor_normalized_df[~neighbor_normalized_df.index.isin(selected_cells.index)].copy()

# seperate neighbor_normalized_df_genes by selected_cells as train and test data
if neighbor_normalized_df_genes is not None:
    train_neighbor_normalized_df_genes = neighbor_normalized_df_genes[neighbor_normalized_df_genes.index.isin(selected_cells.index)].copy()
    test_neighbor_normalized_df_genes = neighbor_normalized_df_genes[~neighbor_normalized_df_genes.index.isin(selected_cells.index)].copy()


In [ ]:
# prototype
# calculate prototype
if neighbor_normalized_df_genes is not None:
    train_neighbor_df = train_neighbor_normalized_df.join(train_neighbor_normalized_df_genes).copy()
else:
    train_neighbor_df = train_neighbor_normalized_df
one_shot_df = pd.concat([selected_cells[config.name_truth].loc[train_neighbor_df.index], train_neighbor_df], axis=1).groupby(config.name_truth, observed=False).mean()
print(one_shot_df.index)

# prompt

In [ ]:
domain_mapping = {0: "Airway Healthy",
                  1: "Airway Inflamed",
                  2: "Large Vessel Healthy",
                  3: "Large Vessel Inflamed",
                  4: "Alveolar",
                  5: "Fibrotic",
                  6: "Immune",
                  7: "Repair"}

config.domain_mapping = domain_mapping


In [ ]:
# set input_df as train_neighbor_normalized_df
if config.Graph_type == "countPlusGenes":
    input_df = train_neighbor_normalized_df
    df_extra = train_neighbor_normalized_df_genes
    sys_prompt_func = prompt.CP_celltype_geneorder
    user_prompt_func = prompt.finetune_user_celltype_geneorder
    assistant_prompt_func = prompt.finetune_assistant
elif config.Graph_type == "count":
    input_df = train_neighbor_normalized_df
    df_extra = None
    sys_prompt_func = prompt.CP_celltype
    user_prompt_func = prompt.finetune_user_celltype
    assistant_prompt_func = prompt.finetune_assistant
elif config.Graph_type == "GeneOnly":
    input_df = train_neighbor_normalized_df_genes
    df_extra = None
    sys_prompt_func = prompt.CP_geneorder
    user_prompt_func = prompt.oneshot_geneorder
    assistant_prompt_func = prompt.finetune_assistant
else:
    raise ValueError(f"Graph_type {config.Graph_type} not supported")

In [ ]:
# generate Comparison-based Prompt
config.system_prompt = sys_prompt_func(one_shot_df, config)

# finetuning with GPT-4o-mini

## creating file

In [ ]:
print(f"finetune_json/{config.data_name}_{config.model_type}/")
print(config.replicate)

In [ ]:
# generate json for finetune
output_folder = f"finetune_json/{config.data_name}_{config.model_type}/"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

train_output_file = f"{output_folder}{config.data_name}_train_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
print(f"Generating json for training into {train_output_file}")
with open(train_output_file, 'w') as f:
    for i in range(train_neighbor_normalized_df.shape[0]):
        system_p = config.system_prompt
        assistant_p = assistant_prompt_func(input_df, i, adata.obs[config.name_truth])
        if config.Graph_type == "countPlusGenes":
            user_p = user_prompt_func(input_df, df_extra, i, config)
        else:
            user_p = user_prompt_func(input_df, i, config)
        
        row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
        json_str = json.dumps(row_data)
        f.write(json_str + '\n') 

## start finetuning

In [ ]:
train_file = client.files.create(
  file=open(train_output_file, "rb"),
  purpose="fine-tune"
)


In [ ]:
finetune_job = client.fine_tuning.jobs.create(
  training_file=train_file.id,
  model="gpt-4o-mini-2024-07-18",
  suffix=f"{config.model_type}_{config.r}" , # default is empty
  # method={
  #       "type": "supervised",
  #       "supervised": {
  #           "hyperparameters": {"n_epochs": 2},
  #       },
  #   }
)

# Use the finetuned model
get the model ID 

IMPORTANT change the config file

In [ ]:
config.gpt_model = "ft:gpt-4o-mini-2024-07-18:whh:finetunepro-230267-slide2-50:CDde6OfW"

In [ ]:
# now use the test data
if config.Graph_type == "countPlusGenes":
    input_df = test_neighbor_normalized_df
    df_extra = test_neighbor_normalized_df_genes
    sys_prompt_func = prompt.CP_celltype_geneorder
    user_prompt_func = prompt.finetune_user_celltype_geneorder
    assistant_prompt_func = prompt.finetune_assistant
elif config.Graph_type == "count":
    input_df = test_neighbor_normalized_df
    df_extra = None
    sys_prompt_func = prompt.CP_celltype
    user_prompt_func = prompt.finetune_user_celltype
    assistant_prompt_func = prompt.finetune_assistant
elif config.Graph_type == "GeneOnly":
    input_df = test_neighbor_normalized_df_genes
    df_extra = None
    sys_prompt_func = prompt.CP_geneorder
    user_prompt_func = prompt.oneshot_geneorder
    assistant_prompt_func = prompt.finetune_assistant
else:
    raise ValueError(f"Graph_type {config.Graph_type} not supported")

In [ ]:
# shorten redundent computation
unique_indices, idx_mapping, inverse_mapping = get_unique_prompts(input_df, 
                                                    config, 
                                                    user_prompt_func, 
                                                    df_extra=df_extra)
input_df = input_df.iloc[unique_indices]
if df_extra is not None:
    df_extra = df_extra.iloc[unique_indices]

In [ ]:
config.replicate = "_all_2niche_50cells"

In [ ]:
# choose correct data and prompt
generate_json_end2end(input_df, 
                      config, 
                      user_prompt_func, 
                      batch_size = 2000,
                      max_completion_tokens = 512,  # key to control the cost, expecially for o3-mini
                      n_rows = 1,
                      df_extra = df_extra)

## submit

In [ ]:
import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_finetunePro_copd.yaml {config.data_name} {config.replicate} > outs/{config.data_name}_finetunePro{config.replicate}.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 4
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']

                # extract outputs - handle both JSON format and text format
                extract_dict = extract_json_microenvironment(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['finetune_gpt4o_mini']

## restore the full labels if use unique_df

In [ ]:
restored_full_df=restore_full_data_from_whole_df(neighbor_normalized_df, 
                                                gpt_results_df, 
                                                config, user_prompt_func, 
                                                df_extra=neighbor_normalized_df_genes)



## Comparison of Zeroshot vs Finetune Results


In [ ]:
# Extract the two columns for comparison
zeroshot_results = adata.obs.loc[restored_full_df.index, 'zeroshot_gpt4o_mini_refined']
finetune_results = restored_full_df['finetune_gpt4o_mini']

# Create a comparison dataframe
comparison_df = pd.DataFrame({
    'zeroshot': zeroshot_results,
    'finetune': finetune_results,
    'index': restored_full_df.index
})

# Add a column to indicate if there was a change
comparison_df['changed'] = comparison_df['zeroshot'] != comparison_df['finetune']

print(f"Total cells: {len(comparison_df)}")
print(f"Cells with changed predictions: {comparison_df['changed'].sum()}")
print(f"Percentage changed: {100 * comparison_df['changed'].mean():.1f}%")


In [ ]:
# Option 1: Confusion Matrix / Heatmap
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Get unique labels from both columns
all_labels = sorted(list(set(comparison_df['zeroshot'].unique()) | set(comparison_df['finetune'].unique())))

# Create confusion matrix
conf_matrix = confusion_matrix(comparison_df['zeroshot'], comparison_df['finetune'], labels=all_labels)

# Plot confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(conf_matrix, 
            xticklabels=all_labels, 
            yticklabels=all_labels,
            annot=True, 
            fmt='d', 
            cmap='Blues',
            cbar_kws={'label': 'Number of cells'})
plt.xlabel('Finetune Results')
plt.ylabel('Zeroshot Results')
plt.title('Confusion Matrix: Zeroshot vs Finetune Predictions')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Calculate and display accuracy
accuracy = np.trace(conf_matrix) / np.sum(conf_matrix)
print(f"Agreement between zeroshot and finetune: {accuracy:.3f} ({100*accuracy:.1f}%)")


In [ ]:
# Option 4: Change summary table and visualization
# Create a cross-tabulation showing all transitions
transition_table = pd.crosstab(comparison_df['zeroshot'], comparison_df['finetune'], margins=True)
print("Transition Table (Zeroshot → Finetune):")
print(transition_table)
print("\n")

# Identify the most common changes
changes_df = comparison_df[comparison_df['changed']].copy()
if len(changes_df) > 0:
    change_summary = changes_df.groupby(['zeroshot', 'finetune']).size().reset_index(name='count')
    change_summary = change_summary.sort_values('count', ascending=False)
    print(f"Most common transitions (top 10):")
    print(change_summary.head(10))
    
    # Plot the most common changes
    if len(change_summary) > 0:
        plt.figure(figsize=(5, 6))
        top_changes = change_summary.head(10)
        labels = [f"{row['zeroshot']} → {row['finetune']}" for _, row in top_changes.iterrows()]
        
        plt.barh(range(len(top_changes)), top_changes['count'])
        plt.yticks(range(len(top_changes)), labels)
        plt.xlabel('Number of cells')
        plt.title('Most Common Label Changes (Zeroshot → Finetune)')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
else:
    print("No changes detected between zeroshot and finetune results!")


In [ ]:
# Option 5: Overlaid histogram showing distribution changes
plt.figure(figsize=(12, 6))

# Get all unique labels
all_unique_labels = sorted(list(set(comparison_df['zeroshot'].unique()) | set(comparison_df['finetune'].unique())))

# Create x positions for bars
x_pos = np.arange(len(all_unique_labels))
width = 0.35

# Get counts for each method
zeroshot_counts = [comparison_df['zeroshot'].value_counts().get(label, 0) for label in all_unique_labels]
finetune_counts = [comparison_df['finetune'].value_counts().get(label, 0) for label in all_unique_labels]

# Create bars
bars1 = plt.bar(x_pos - width/2, zeroshot_counts, width, label='Zeroshot', alpha=0.7, color='skyblue')
bars2 = plt.bar(x_pos + width/2, finetune_counts, width, label='Finetune', alpha=0.7, color='lightcoral')

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    if height > 0:
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{int(height)}', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    if height > 0:
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{int(height)}', ha='center', va='bottom', fontsize=9)

plt.xlabel('Cell Types')
plt.ylabel('Count')
plt.title('Distribution Comparison: Zeroshot vs Finetune Predictions')
plt.xticks(x_pos, all_unique_labels, rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

# Calculate the net change for each cell type
net_changes = pd.DataFrame({
    'cell_type': all_unique_labels,
    'zeroshot_count': zeroshot_counts,
    'finetune_count': finetune_counts
})
net_changes['net_change'] = net_changes['finetune_count'] - net_changes['zeroshot_count']
net_changes['percent_change'] = ((net_changes['finetune_count'] - net_changes['zeroshot_count']) / 
                                net_changes['zeroshot_count'].replace(0, 1) * 100)

print("Net changes in cell type counts:")
print(net_changes.sort_values('net_change', key=abs, ascending=False))


In [ ]:
import re
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.colors as mcolors

def convert_color_to_rgba(color, alpha=0.5):
    """Robustly convert many color formats (hex, 'rgb(...)', named) -> 'rgba(r,g,b,a)'."""
    if color is None:
        return f"rgba(0,0,0,{alpha})"
    color = str(color).strip()

    # Match rgb(...) or rgba(...)
    m = re.match(r'rgba?\(\s*(\d{1,3})\s*,\s*(\d{1,3})\s*,\s*(\d{1,3})(?:\s*,\s*([0-9.]+))?\s*\)', color, flags=re.I)
    if m:
        r, g, b = int(m.group(1)), int(m.group(2)), int(m.group(3))
        # clamp
        r, g, b = max(0, min(255, r)), max(0, min(255, g)), max(0, min(255, b))
        if m.group(4) is not None:
            original_alpha = float(m.group(4))
            # combine original alpha and requested alpha (multiply so transparent sources stay transparent)
            final_alpha = max(0.0, min(1.0, original_alpha * alpha))
        else:
            final_alpha = alpha
        return f"rgba({r},{g},{b},{final_alpha})"

    # Hex colors: #rgb or #rrggbb
    if color.startswith('#'):
        hexc = color.lstrip('#')
        if len(hexc) == 3:
            hexc = ''.join([c*2 for c in hexc])
        if len(hexc) == 6:
            try:
                r = int(hexc[0:2], 16)
                g = int(hexc[2:4], 16)
                b = int(hexc[4:6], 16)
                return f"rgba({r},{g},{b},{alpha})"
            except ValueError:
                pass

    # Finally try matplotlib named-color parsing
    try:
        r, g, b = mcolors.to_rgb(color)  # returns floats 0..1
        return f"rgba({int(r*255)},{int(g*255)},{int(b*255)},{alpha})"
    except Exception:
        # fallback: black with requested alpha
        return f"rgba(0,0,0,{alpha})"


def create_enhanced_sankey(df, source_col, target_col):
    """
    Create an enhanced Sankey diagram with better positioning and styling
    """
    # Get transition data
    transition_counts = df.groupby([source_col, target_col]).size().reset_index(name='count')

    # Create separate lists for source and target labels
    source_labels = df[source_col].unique().tolist()
    target_labels = df[target_col].unique().tolist()

    # Create node labels
    node_labels = [f"{label}" for label in source_labels] + [f"{label}" for label in target_labels]

    # Mapping to node indices
    source_to_idx = {label: idx for idx, label in enumerate(source_labels)}
    target_to_idx = {label: idx + len(source_labels) for idx, label in enumerate(target_labels)}

    # Prepare Sankey data
    source_indices = [source_to_idx[label] for label in transition_counts[source_col]]
    target_indices = [target_to_idx[label] for label in transition_counts[target_col]]
    values = transition_counts['count'].tolist()

    # Number of nodes
    num_source = len(source_labels)
    num_target = len(target_labels)

    # Evenly distribute y for source and target separately (no overlap)
    source_y = [i/(num_source-1) if num_source > 1 else 0.5 for i in range(num_source)]
    target_y = [i/(num_target-1) if num_target > 1 else 0.5 for i in range(num_target)]

    # Assign x/y coordinates (left/right columns). Slightly nudge x positions to avoid exact overlap with edges.
    node_x = [0.05] * num_source + [0.95] * num_target
    node_y = source_y + target_y

    # Colors
    source_colors = px.colors.qualitative.Set1[:num_source] if num_source > 0 else []
    target_colors = px.colors.qualitative.Set2[:num_target] if num_target > 0 else []
    all_colors = source_colors + target_colors

    # Link colors (based on source color, transparent)
    link_colors = []
    for src_idx in source_indices:
        # src_idx refers to source node index (0..num_source-1)
        base_color = source_colors[src_idx] if src_idx < len(source_colors) else None
        link_colors.append(convert_color_to_rgba(base_color, 0.25))

    return (node_labels, node_x, node_y, all_colors,
            source_indices, target_indices, values, link_colors)


# --- Example usage ---
(node_labels, node_x, node_y, node_colors,
 source_indices, target_indices, values, link_colors) = create_enhanced_sankey(
    comparison_df, 'zeroshot', 'finetune'
)

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=7,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=node_colors,
        # x=node_x,
        # y=node_y,
        # hovertemplate='<b>%{label}</b><br>Total flow: %{value}<extra></extra>'
    ),
    link=dict(
        source=source_indices,
        target=target_indices,
        value=values,
        color=link_colors,
        # hovertemplate='<b>%{source.label}</b><br>→ <b>%{target.label}</b><br>Count: %{value} cells<extra></extra>'
    )
)])

fig.update_layout(
    title_text="Cell Type Prediction Changes: Zeroshot → Finetune<br><sub>Width of flows proportional to number of cells</sub>",
    font_size=7,
    width=500,
    height=400,
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()



## plot and refine

In [ ]:
adata.obs = adata.obs.join(restored_full_df)
adata.obs['selection'] = False


In [ ]:
adata.obs.loc[adata.obs['finetune_gpt4o_mini'].isna(), 'selection'] = True

In [ ]:
adata.obs.loc[adata.obs['selection'], 'finetune_gpt4o_mini'] = adata.obs.loc[adata.obs['selection'], 'zeroshot_gpt4o_mini_refined']

In [ ]:
#refine the finetune_gpt4o_mini
adata.obs['finetune_gpt4o_mini_refined'] = relabel_cells(adj_matrix.toarray(), adata.obs['finetune_gpt4o_mini'])

In [ ]:
sc.set_figure_params(figsize=(5, 5))
# set colors
adata.uns['finetune_gpt4o_mini_refined_colors'] = ['#e377c2', '#ff7f0e', '#8c564b', '#d62728', '#9467bd', '#2ca02c', '#1f77b4']
sc.pl.scatter(adata, x="y", y="x", 
color="finetune_gpt4o_mini_refined", 
title =  f"finetune_gpt4o_mini_refined",
)

In [ ]:
sc.set_figure_params(figsize=(5, 6), dpi_save=300)
# set colors
adata.uns['zeroshot_gpt4o_mini_refined_colors'] = ['#e377c2',
  '#ff7f0e',
  '#8c564b',
  '#d62728',
  '#7f7f7f',
  '#9467bd',
  '#2ca02c',
  '#1f77b4']
sc.pl.scatter(adata, x="y", y="x", color="zeroshot_gpt4o_mini_refined", title =  f"zeroshot_gpt4o_mini_refined",
)

## save

In [ ]:
print(f"{data_path}/{config.data_name}_finetune_gpt4o_refined_k20.csv")

In [ ]:
adata.obs.to_csv(f"{data_path}/{config.data_name}_finetune_gpt4o_refined_k20.csv")

# plot for paper

In [ ]:
adata.obs = pd.read_csv(f"{data_path}/{config.data_name}_finetune_gpt4o_refined_k20.csv")

In [ ]:
import seaborn as sns

def plot_scatter(adata, label='zeroshot_gpt4o_mini_refined'):
    """
    Draws a scatter plot using adata.obs data.
    
    Parameters:
    adata (AnnData): The annotated data object containing .obs dataframe.
    """
    
    # Setup figure
    fig, ax = plt.subplots(figsize=(4, 3))
    
    # Custom colors provided by user
    label_color = ['#e377c2',
        '#ff7f0e',
        '#8c564b',
        '#d62728',
        '#7f7f7f',
        '#9467bd',
        '#2ca02c',
        '#1f77b4']
    label_name = ['Airway Inflamed', 'Alveolar', 'Fibrotic', 'Immune', 'Inflamed', 'Large Vessel Healthy','Large Vessel Inflamed',   'Repair']
    color_dict = dict(zip(label_name, label_color))
    
    # Helper to check if column exists and mapped correctly
    if label not in adata.obs.columns:
        print("Error: Column 'zeroshot_gpt4o_mini_refined' not found in adata.obs")
        return

    # Create scatter plot
    # Using hue_order if categories are known improves stability, but simpler here.
    sns.scatterplot(
        data=adata.obs,
        x='y',
        y='x',
        hue=label,
        palette=color_dict,
        s=1,          # Adjust dot size
        linewidth=0,   # Remove white edge around points for cleaner look
        ax=ax
    )
    
    # Scientific style adjustments
    
    # 1. No axis labels
    ax.set_xlabel('')
    ax.set_ylabel('')
    
    # 2. No marks (ticks)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # 3. Plain background
    ax.set_facecolor('white')
    fig.patch.set_facecolor('white')
    
    # Add a black frame around the plot
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(1)
        
    # 4. Legend font size 7
    # Move legend outside if too crowded, or keep inside. 
    # Usually scientific plots might put it outside.
    plt.legend(
        fontsize=7, 
        title='', 
        frameon=False, 
        loc='center left', 
        bbox_to_anchor=(1, 0.5),
        markerscale=4
    )
    
    plt.tight_layout()
    plt.savefig(f"figures/{config.data_name}_{label}.pdf")
    plt.show()

# Usage example (copy to notebook):
# plot_scatter(adata)


In [ ]:
plot_scatter(adata, label="finetune_gpt4o_mini_refined")


In [ ]:
def plot_pie_chart(adata, config, label_name, label_color):
    """
    Plots a pie chart for the 'finetune_gpt4o_mini_refined' column 
    for cells where config.celltype_name is 'SMC'.
    
    Parameters:
    adata: AnnData object
    config: Configuration object (must have celltype_name attribute)
    label_name: List of label names for colors
    label_color: List of colors corresponding to label_name
    """
    
    # 1. Filter data for SMC
    # Ensure config.celltype_name exists and is valid
    if not hasattr(config, 'celltype_name'):
        print("Error: config object missing 'celltype_name' attribute")
        return
        
    celltype_col = config.celltype_name
    if celltype_col not in adata.obs.columns:
        print(f"Error: Column '{celltype_col}' not found in adata.obs")
        return

    subset = adata.obs[adata.obs[celltype_col] == "SMC"]
    
    if subset.empty:
        print("Warning: No cells found with SMC label.")
        return

    # 2. Get value counts for the target column
    target_col = "finetune_gpt4o_mini_refined"
    if target_col not in subset.columns:
        print(f"Error: Column '{target_col}' not found in adata.obs")
        return
        
    counts = subset[target_col].value_counts()
    
    # 3. Prepare colors
    color_dict = dict(zip(label_name, label_color))
    # Map colors to the indices of counts. Use a default if not found.
    pie_colors = [color_dict.get(label, '#cccccc') for label in counts.index]
    
    # 4. Plot
    fig, ax = plt.subplots(figsize=(3, 3))
    
    # Simple, clean scientific style
    wedges, texts, autotexts = ax.pie(
        counts, 
        labels=counts.index, 
        colors=pie_colors, 
        autopct='%1.1f%%', 
        startangle=90,
        textprops=dict(color="black", fontsize=7)
    )
    
    ax.set_title(f"Distribution of {target_col}\n(SMC Cells)", fontsize=7)
    
    plt.tight_layout()
    plt.savefig("figures/pie_chart.pdf")
    plt.show()

In [ ]:
plot_pie_chart(adata, config, label_name, label_color)


In [ ]:
# Convert both columns to strings
adata.obs['zeroshot_gpt4o_mini_refined'] = adata.obs['zeroshot_gpt4o_mini_refined'].astype(str)
adata.obs['finetune_gpt4o_mini_refined'] = adata.obs['finetune_gpt4o_mini_refined'].astype(str)
# Create comparison_df
comparison_df = pd.DataFrame({
    'zeroshot': adata.obs['zeroshot_gpt4o_mini_refined'],
    'finetune': adata.obs['finetune_gpt4o_mini_refined']
}, index=adata.obs.index)
# Add changed column
comparison_df['changed'] = comparison_df['zeroshot'] != comparison_df['finetune']


# end